---
---
---

# Energy Model

---
---
---

<br>

> **_Abstract:_** ...

> **_Table of Contents:_**
>
> 1. DataFrame
> 2. Training
>     - Train-Test Split
>     - Final Model
> 3. Testing
> 4. Artifacts
> 5. Conclusion

<br>

> | **Language: python@3.11** |
> | - |

<br>

> | **_Source:_** [**_EnergyPlus AI: Energy Model_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-energy-model) |
> | - |

<br>

> | **_Libraries:_** [**_github.com/robertovicario/EnergyPlus-AI/notebook/lib_**](https://github.com/robertovicario/EnergyPlus-AI/tree/main/notebook/lib) |
> | - |

<br>

---
---
---

## Dependencies

### Packages

In [1]:
from datetime import datetime
from joblib import dump, load
from loguru import logger
from scipy.stats import randint, loguniform
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split
import json
import numpy as np
import os
import pandas as pd
import warnings

from lib import ml_metrics as metrics_lib

# -------------------------

warnings.filterwarnings('ignore')

### Paths

In [2]:
DATA_PATH = os.path.join('..', 'data')
MODELS_PATH = os.path.join('..', 'models')

### Configurations

In [3]:
"""
> (!) WARNING
>
> Set `tuning = True` to enable hyperparameter tuning, `False` otherwise.
"""

tuning = False

In [4]:
config = {
    'model': {
        'name': 'Energy_Model',
        'task': 'Regression',
        'dataset': 'udse.parquet',
        'target': 'EUI'
    },
    'training': {
        'df_rows': 0,
		'df_columns': 0,
        'frac': 1.0,
        'test_size': 0.2,
        'n_jobs': -1,
        'random_state': 42,
        'shuffle': True
    }
}

## DataFrame

> **_Source:_** [**_EnergyPlus AI: Training DataFrame_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-training-dataframe)

In [5]:
df = pd.read_parquet(
    os.path.join(DATA_PATH, 'udse.parquet'),
    engine='fastparquet'
)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 255036 entries, 0 to 260747
Data columns (total 23 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   Building-Type    255036 non-null  category
 1   Climate-Zone     255036 non-null  category
 2   Total-Area       255036 non-null  float64 
 3   Floor-Area       255036 non-null  float64 
 4   Num-Floors       255036 non-null  float64 
 5   Building-Depth   255036 non-null  float64 
 6   Building-Length  255036 non-null  float64 
 7   Building-Height  255036 non-null  float64 
 8   Floor-Height     255036 non-null  float64 
 9   WWR              255036 non-null  float64 
 10  Wall-R-Value     255036 non-null  category
 11  Roof-R-Value     255036 non-null  category
 12  Window-U-Value   255036 non-null  category
 13  SHGC             255036 non-null  category
 14  EUI              255036 non-null  float64 
 15  HVAC-Type        255036 non-null  category
 16  HVAC-AC          255036 n

## Training

### Train-Test Split

In [6]:
# Feature Selection
X = df.drop(columns=['EUI'])
y = df['EUI']

# Fractional Sampling
frac = config['training']['frac']
subset = int(len(X) * frac)
X = X.iloc[:subset]
y = y.iloc[:subset]
config['training']['df_rows'] = df.shape[0]
config['training']['df_columns'] = df.shape[1]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['training']['test_size'],
    random_state=config['training']['random_state'],
    shuffle=config['training']['shuffle']
)

# -------------------------

logger.debug(f"DF{'':<4}(Shape): {X.shape}")
logger.debug(f"TRAIN{'':<1}(Shape): {X_train.shape}")
logger.debug(f"TEST{'':<2}(Shape): {X_test.shape}")

2026-04-27 11:53:31.035 | DEBUG    | __main__:<module>:23 - DF    (Shape): (255036, 22)
2026-04-27 11:53:31.035 | DEBUG    | __main__:<module>:24 - TRAIN (Shape): (204028, 22)
2026-04-27 11:53:31.036 | DEBUG    | __main__:<module>:25 - TEST  (Shape): (51008, 22)


### Hyperparameter Tuning

In [7]:
estimator = HistGradientBoostingRegressor(
    categorical_features='from_dtype',
    random_state=config['training']['random_state']
)
model_params = {
    "l2_regularization": 8.485420586212651,
    "learning_rate": 0.047006594771349115,
    "max_iter": 301,
    "max_leaf_nodes": 25,
    "min_samples_leaf": 52
}

In [8]:
if tuning:
    param_distributions = {
        'learning_rate': loguniform(0.01, 0.2),
        'max_iter': randint(300, 900),
        'max_leaf_nodes': randint(25, 100),
        'min_samples_leaf': randint(5, 100),
        'l2_regularization': loguniform(1e-3, 10)
    }
    search = RandomizedSearchCV(
        estimator,
        param_distributions=param_distributions,
        n_iter=50,
        scoring='neg_mean_squared_error',
        n_jobs=config['training']['n_jobs'],
        cv=3,
        random_state=config['training']['random_state']
    )
    search.fit(X_train, y_train)
    logger.debug(f"Best Params: {search.best_params_}")
    logger.debug(f"Best Score: {search.best_score_}")
    model_params = search.best_params_

### Final Model

In [9]:
model = TransformedTargetRegressor(
    regressor=HistGradientBoostingRegressor(
        **model_params,
        categorical_features='from_dtype',
        random_state=config['training']['random_state']
    ),
    func=np.log1p,
    inverse_func=np.expm1
)
model.fit(X_train, y_train)
display(model)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",HistGradientB...ndom_state=42)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.047006594771349115
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",301
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",25
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None


## Testing

In [10]:
# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Metrics
metrics_out = {}
display(df['EUI'].describe())
metrics_lib.compute_metrics_reg(metrics_out, y_train, y_pred_train, set='TRAIN')
metrics_lib.compute_metrics_reg(metrics_out, y_test, y_pred_test, set='TEST')
metrics_lib.out_metrics(metrics_out, ml_task='regression')

count    255036.000000
mean        132.211597
std         162.203232
min          18.086152
25%          45.613598
50%          72.962024
75%         100.841246
max        1070.272026
Name: EUI, dtype: float64

,Size,MAE,RMSE,R2
TRAIN,204028.0,14.329712,35.793094,0.951454
TEST,51008.0,14.166444,35.202108,0.952315


## Artifacts

In [ ]:
# Timestamp
timestamp = datetime.now().strftime("%Y-%b-%d-%H:%M:%S")
artifacts_dir = os.path.join(MODELS_PATH, timestamp)
os.makedirs(artifacts_dir, exist_ok=True)

# Model
from joblib import dump, load
dump(model, os.path.join(artifacts_dir, 'model.joblib'))
display(load(os.path.join(artifacts_dir, 'model.joblib')))

# Artifacts
metadata = {
    'metadata': {
        'timestamp': timestamp
    },
    'config': config,
    'hyperparameters': model_params,
    'metrics': {
		k: dict(zip(['Size', 'MAE', 'RMSE', 'R2'], v))
		for k, v in metrics_out.items()
	}
}
with open(os.path.join(artifacts_dir, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)
print(json.dumps(metadata, indent=4))

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",HistGradientB...ndom_state=42)
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.047006594771349115
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",301
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",25
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None


{
    "metadata": {
        "timestamp": "2026-Apr-27-11:53:35"
    },
    "config": {
        "model": {
            "name": "Energy_Model",
            "task": "Regression",
            "dataset": "udse.parquet",
            "target": "EUI"
        },
        "training": {
            "df_rows": 255036,
            "df_columns": 23,
            "frac": 1.0,
            "test_size": 0.2,
            "n_jobs": -1,
            "random_state": 42,
            "shuffle": true
        }
    },
    "hyperparameters": {
        "l2_regularization": 8.485420586212651,
        "learning_rate": 0.047006594771349115,
        "max_iter": 301,
        "max_leaf_nodes": 25,
        "min_samples_leaf": 52
    },
    "metrics": {
        "TRAIN": {
            "Size": 204028,
            "MAE": 14.329711580923712,
            "RMSE": 35.7930941837863,
            "R2": 0.951454022507507
        },
        "TEST": {
            "Size": 51008,
            "MAE": 14.16644396506598,
            "RMSE": 3

## Conclusion

> ...

> Further experimentation and model development can be found in the following notebooks:
>
> - [**_EnergyPlus AI: Energy Model_**](https://www.kaggle.com/code/robertovicario/energyplus-ai-energy-model)